# 05 · Export to Hugging Face Hub

Uploads your fine-tuned model (`merged/` folder) to Hugging Face, so anyone can load it with **one line**:
`AutoModelForCausalLM.from_pretrained("your-name/your-model")`.

**Steps:** 1. config → 2. check the files → 3. test it once → 4. write a model card → 5. log in → 6. upload.

You need a free Hugging Face account and an access token with **write** permission (huggingface.co → Settings → Access Tokens).

## 1. Config
Set `HF_REPO_ID` to `your-hf-username/model-name`. Keep `PRIVATE = True` until you've checked the upload; you can make it public later on the website.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"

RUN_NAME = "sft_qwen2.5-0.5b"            # change to "sft_qwen2.5-0.5b_opt" after the optimized run
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_DIR = NOTEBOOKS / "data" / "eval" / "results" / RUN_NAME

HF_REPO_ID = "your-hf-username/qwen2.5-0.5b-toolcaller"   # <-- change this
PRIVATE = True

print("Model folder:", MERGED_DIR, "| exists:", MERGED_DIR.exists())

## 2. Check what we're uploading
The folder must contain the weights (`model.safetensors`), the architecture (`config.json`) and the tokenizer. The safetensors header tells us the number type and the parameter count without loading the model.

In [ ]:
import math
import struct

for f in sorted(MERGED_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1024**2:8.1f} MB")

with open(MERGED_DIR / "model.safetensors", "rb") as fh:
    header = json.loads(fh.read(struct.unpack("<Q", fh.read(8))[0]))
header.pop("__metadata__", None)
n_params = sum(math.prod(t["shape"]) for t in header.values())
dtypes = {t["dtype"] for t in header.values()}
print(f"\n{len(header)} tensors | {n_params / 1e6:.0f}M parameters | dtype: {dtypes}")

## 3. Test it once before uploading
Load with plain `transformers` and ask one question in the **trained prompt format** (`build_messages`: system prompt + tools + request). We expect a JSON tool call back.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.prompts import build_messages
from src.evaluate import extract_json

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype="auto", device_map="auto")

record = {
    "query": "What's the weather like in Pune right now?",
    "tools": [{"name": "get_weather", "description": "Current weather for a city.",
               "parameters": {"city": {"type": "str", "description": "City name"}}}],
}
ids = tokenizer.apply_chat_template(build_messages(record), add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=128, do_sample=False)
answer = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print("MODEL :", answer)
print("PARSED:", extract_json(answer))

del model
torch.cuda.empty_cache()

## 4. Write the model card (`README.md`)
The model card is the page people see on Hugging Face. The block at the top (between `---`) is metadata: base model, license and tags make it searchable. The **prompt format** section matters most: without it, nobody can use the model correctly. Results are read from your `metrics.json`, so they're never typed by hand.

In [ ]:
from src.prompts import SYSTEM_PROMPT

m = json.loads((RESULTS_DIR / "metrics.json").read_text())
s, vs = m["summary"], m["vs_base"]
row = lambda k: f"| {k} | {s[k]['value']:.1f}% | {s[k]['ci_low']:.1f}–{s[k]['ci_high']:.1f} |"

card = f"""---
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
tags: [function-calling, tool-use, qlora, unsloth]
datasets: [Salesforce/xlam-function-calling-60k]
---

# {HF_REPO_ID.split("/")[-1]}

Qwen2.5-0.5B-Instruct fine-tuned with **QLoRA (Unsloth)** for **function calling**: given a list of tools and a request,
it returns the tool call(s) as a JSON array.

## Results (frozen test set, {m["n"]:,} questions, greedy decoding)
| Metric | Fine-tuned | 95% CI |
|---|---|---|
{row("exact_match")}
{row("function_match")}
{row("argument_f1")}
{row("schema_valid")}

Base model exact match: **{vs["a"]}%** → fine-tuned: **{vs["b"]}%** (McNemar p = {vs["mcnemar_p"]}).

## Prompt format (important)
The model was trained on this exact format. Plain chat without tools gets a normal text answer.

**System:**
```
{SYSTEM_PROMPT.strip()}
```
**User:**
```
Available tools:

<tools as a JSON list>

User request:

<the request>

Return the required tool call(s) as JSON.
```
**Output:** `[{{"name": "get_weather", "arguments": {{"city": "Pune"}}}}]`

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", dtype="auto", device_map="auto")
# SYSTEM_PROMPT / USER_PROMPT = the two blocks shown above
messages = [{{"role": "system", "content": SYSTEM_PROMPT}}, {{"role": "user", "content": USER_PROMPT}}]
ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
print(tok.decode(model.generate(ids, max_new_tokens=256, do_sample=False)[0][ids.shape[1]:], skip_special_tokens=True))
```

## Training
QLoRA r={m["training"]["lora_r"]}, alpha={m["training"]["lora_alpha"]}, all linear layers, {m["training"]["epochs"]} epoch,
effective batch {m["training"]["effective_batch"]}, lr {m["training"]["learning_rate"]}, max length {m["training"]["max_seq_length"]},
loss on the answer only. LoRA merged into the bf16 base.

## Limitations
- Trained on single-turn tool selection: it picks the call, it doesn't read tool results or write a final answer.
- Custom prompt format, not Qwen's native `<tool_call>` format, so `tools=` APIs need a small wrapper.
"""
(MERGED_DIR / "README.md").write_text(card, encoding="utf-8")
print(card[:1500])

## 5. Log in
Paste your **write** token when the box appears. It's saved on your PC, so you only do this once.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 6. Upload
`create_repo` makes the (private) repo, and `upload_folder` sends the whole `merged/` folder. Safetensors files are uploaded in chunks, so ~1 GB takes a few minutes.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_ID, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=str(MERGED_DIR), repo_id=HF_REPO_ID,
                  commit_message=f"Upload {RUN_NAME} (merged QLoRA)")
print(f"✅ Done: https://huggingface.co/{HF_REPO_ID}")

## 7. Check: load it back from the Hub
This downloads the model the way any user would and runs the same test as step 3. If it prints a tool call, the upload works.

In [ ]:
tok_hub = AutoTokenizer.from_pretrained(HF_REPO_ID)
model_hub = AutoModelForCausalLM.from_pretrained(HF_REPO_ID, dtype="auto", device_map="auto")
ids = tok_hub.apply_chat_template(build_messages(record), add_generation_prompt=True, return_tensors="pt").to(model_hub.device)
print(tok_hub.decode(model_hub.generate(ids, max_new_tokens=128, do_sample=False)[0][ids.shape[1]:], skip_special_tokens=True))